<a href="https://colab.research.google.com/github/Anthea05/Anthea_machine-learning/blob/main/JS09/Tugas1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install faiss-cpu annoy hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 6.2 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 27.6 MB/s  0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520956 sha256=f57caaca79a52b55737c582b38b4275a0ab469a2a039e10b017766e7ff1c4350
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679066 sha256=6b2f1a85c50b7cd295bbe1219a9358f39f26375001de23186f4592b166fbf9fc
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50

In [4]:
import csv
import time
import numpy as np
import pandas as pd
import faiss
import hnswlib
from annoy import AnnoyIndex
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# 1. Load Data
path = '/content/songs_with_attributes_and_lyrics.csv'

# Intip header dengan menambahkan quoting=3 agar tanda kutip tidak merusak pemisahan kolom
kolom_file = pd.read_csv(path, nrows=0, quoting=3).columns.tolist()
print("Kolom di file:", kolom_file)

# Tentukan separator
sep = ';' if (len(kolom_file) == 1 and ';' in kolom_file[0]) else ','

def load_csv_aman(path, sep):
    # Cara 1: pandas biasa (cepat)
    try:
        return pd.read_csv(path, sep=sep, on_bad_lines='skip', quoting=3)
    except Exception as e:
        print("pandas gagal baca file:", e)
        print("Pindah ke pembaca manual...")

    csv.field_size_limit(10**9)
    rows, dilewati = [], 0
    with open(path, 'r', encoding='utf-8-sig', errors='replace', newline='') as f:
        reader = csv.reader(f, delimiter=sep)
        header = next(reader)
        for row in reader:
            if len(row) == len(header):
                rows.append(row)
            else:
                dilewati += 1
    print(f"Baris kebaca: {len(rows)}, baris dilewati: {dilewati}")
    return pd.DataFrame(rows, columns=header)


df = load_csv_aman(path, sep)

# Rapihin nama kolom: buang kutip dan spasi, samain jadi huruf kecil
df.columns = (df.columns.astype(str)
              .str.replace(r'["\']', '', regex=True)
              .str.strip()
              .str.lower())

print("Ukuran data awal:", df.shape)

# 2. Ambil fitur numeriknya saja sesuai soal
features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

# Cek dulu kolomnya ada semua atau nggak, kalau nggak ada langsung kelihatan kolom aslinya
missing = [c for c in features if c not in df.columns]
if missing:
    raise ValueError(f"Kolom ini nggak ada di dataset: {missing}\n"
                     f"Kolom yang tersedia: {df.columns.tolist()}")

# Paksa jadi numerik, baris yang berantakan jadi NaN lalu di-drop
df[features] = df[features].apply(pd.to_numeric, errors='coerce')
df = df.dropna(subset=features).reset_index(drop=True)
print("Ukuran data setelah dibersihkan:", df.shape)
X = df[features].values

# 3. Standarisasi data pakai StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype('float32')  # dijadiin float32 buat FAISS

k = 10  # Kita cari 10 lagu yang paling mirip

# Ambil 1 lagu secara acak buat dijadiin query
idx_lagu_dicari = np.random.randint(0, len(X_scaled))
query_1d = X_scaled[idx_lagu_dicari]
query_2d = query_1d.reshape(1, -1)  # Sklearn sama FAISS mintanya bentuk array 2D

# Memastikan kolom 'track_name' ada sebelum print
track_col = 'track_name' if 'track_name' in df.columns else df.columns[0]
print(f"Lagu yang kita cari miripnya: '{df.loc[idx_lagu_dicari, track_col]}' (Index: {idx_lagu_dicari})")
print("-" * 50)

# ==========================================
# A. Exact NN (Sklearn Nearest Neighbors)
# ==========================================
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)

start = time.time()
distances_nn, indices_nn = nn.kneighbors(query_2d)
waktu_nn = time.time() - start

print("\n1. Hasil Exact NN (Scikit-Learn)")
print(f"Waktu Query: {waktu_nn:.6f} detik")
print("Index lagu terdekat:", indices_nn[0])

# ==========================================
# B. ANNOY
# ==========================================
dim = X_scaled.shape[1]
annoy_idx = AnnoyIndex(dim, 'euclidean')

# masukin data satu-satu ke annoy
for i, v in enumerate(X_scaled):
    annoy_idx.add_item(i, v)
annoy_idx.build(10)

start = time.time()
# annoy nerima input vektor 1D
indices_annoy = annoy_idx.get_nns_by_vector(query_1d, k)
waktu_annoy = time.time() - start

print("\n2. Hasil ANNOY")
print(f"Waktu Query: {waktu_annoy:.6f} detik")
print("Index lagu terdekat:", indices_annoy)

# ==========================================
# C. FAISS (IndexFlatL2)
# ==========================================
faiss_idx = faiss.IndexFlatL2(dim)
faiss_idx.add(X_scaled)

start = time.time()
distances_faiss, indices_faiss = faiss_idx.search(query_2d, k)
waktu_faiss = time.time() - start

print("\n3. Hasil FAISS")
print(f"Waktu Query: {waktu_faiss:.6f} detik")
print("Index lagu terdekat:", indices_faiss[0])

# ==========================================
# D. HNSW (hnswlib)
# ==========================================
hnsw_idx = hnswlib.Index(space='l2', dim=dim)
hnsw_idx.init_index(max_elements=len(X_scaled), ef_construction=100, M=16)
hnsw_idx.add_items(X_scaled)
hnsw_idx.set_ef(50)

start = time.time()
indices_hnsw, distances_hnsw = hnsw_idx.knn_query(query_2d, k=k)
waktu_hnsw = time.time() - start

print("\n4. Hasil HNSW")
print(f"Waktu Query: {waktu_hnsw:.6f} detik")
print("Index lagu terdekat:", indices_hnsw[0])

# Opsional: nampilin judul lagu hasilnya dari FAISS aja sebagai contoh
print("\n--- Contoh Daftar Lagu Paling Mirip (dari FAISS) ---")
artist_col = 'track_artist' if 'track_artist' in df.columns else (df.columns[1] if len(df.columns) > 1 else track_col)
for i, idx in enumerate(indices_faiss[0]):
    name = df.loc[idx, track_col] if track_col in df.columns else 'Unknown'
    artist = df.loc[idx, artist_col] if artist_col in df.columns else 'Unknown'
    print(f"{i+1}. {name} - {artist}")

Kolom di file: ['"id"', '"name"', '"album_name"', '"artists"', '"danceability"', '"energy"', '"key"', '"loudness"', '"mode"', '"speechiness"', '"acousticness"', '"instrumentalness"', '"liveness"', '"valence"', '"tempo"', '"duration_ms"', '"lyrics"']


/tmp/ipykernel_2733/4263883945.py:24: DtypeWarning: Columns (10,11,12,13,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, sep=sep, on_bad_lines='skip', quoting=3)


Ukuran data awal: (3180407, 17)
Ukuran data setelah dibersihkan: (3107, 17)
Lagu yang kita cari miripnya: '"23 (Like the Blink-182 Song' (Index: 355)
--------------------------------------------------

1. Hasil Exact NN (Scikit-Learn)
Waktu Query: 0.001619 detik
Index lagu terdekat: [ 355 1258 2391 1928  318 2481 2804 2802 2803 2152]

2. Hasil ANNOY
Waktu Query: 0.000238 detik
Index lagu terdekat: [355, 1258, 2391, 1928, 318, 2481, 2802, 2803, 2804, 2152]

3. Hasil FAISS
Waktu Query: 0.000350 detik
Index lagu terdekat: [ 355 1258 2391 1928  318 2481 2802 2803 2804 2152]

4. Hasil HNSW
Waktu Query: 0.000341 detik
Index lagu terdekat: [ 355 1258 2391 1928  318 2481 2802 2803 2804 2152]

--- Contoh Daftar Lagu Paling Mirip (dari FAISS) ---
1. "23 (Like the Blink-182 Song -  but Without the Homophobia)"
2. "Air Mail Special" - ""
3. "As' Tin Na Leei / Ypo To Miden -  Pt. 1 / Na Pethanoun Oi Gynaikes - Live"
4. "An einem Sonntag in Avignon" - "Herzlichst
5. "2 Veces" - ""
6. "Attracted Woma